# Ensemble member — ConvNeXt-V2-tiny (ImageNet-22k pretrained, seed 11)

One member of a multi-model ensemble trained on **RAF-DB + FER2013** (domain-balanced sampling, grayscale /
low-resolution augmentation). Uses the same train/validation/test splits as every other member, and saves its
logits on every evaluation set so the members can be combined in a separate ensemble notebook.
**CK+ is never used for training.**

## 1 · Data

In [ ]:
import os, sys, json, time, copy, random, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
sys.path.insert(0, os.getcwd())

ARCH, SEED, SPLIT_SEED = 'convnext', 11, 42      # SPLIT_SEED is shared by all members -> identical splits
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DATA = os.environ.get('DATA_ROOT', '/kaggle/input')
OUT  = os.environ.get('OUT_ROOT', '/kaggle/working'); os.makedirs(OUT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CFG = {'ddamfn':   dict(size=112, epochs=10, lr=1e-4, batch=128),
       'convnext': dict(size=224, epochs=14, lr=1e-4, batch=96)}[ARCH]
CFG['epochs'] = int(os.environ.get('EPOCHS', CFG['epochs'])); CFG['workers'] = int(os.environ.get('WORKERS', 4))
S = CFG['size']
CLASS_NAMES = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']
FOLDER2EMO = {'angry': 'Anger', 'anger': 'Anger', 'disgust': 'Disgust', 'fear': 'Fear', 'happy': 'Happiness',
              'happiness': 'Happiness', 'neutral': 'Neutral', 'sad': 'Sadness', 'sadness': 'Sadness',
              'surprise': 'Surprise'}

train_csv = test_csv = ckpt_path = None; IMG = {}; FER_TR, FER_TE, CK = [], [], []
for root, _, files in os.walk(DATA):
    low = root.lower().replace('\\', '/'); parent = os.path.basename(root).lower()
    for f in files:
        p = os.path.join(root, f)
        if   f == 'train_labels.csv': train_csv = p
        elif f == 'test_labels.csv':  test_csv = p
        elif f in ('ddamfn_rafdb_final.pth', 'ddamfn_rafdb_best.pth'): ckpt_path = p
        elif f.lower().endswith(('.jpg', '.jpeg', '.png')):
            emo = FOLDER2EMO.get(parent)
            if emo and 'fer' in low and '/test' in low:    FER_TE.append((p, CLASS_NAMES.index(emo)))
            elif emo and 'fer' in low and '/train' in low: FER_TR.append((p, CLASS_NAMES.index(emo)))
            elif emo and 'ck' in low:                      CK.append((p, CLASS_NAMES.index(emo)))
            else:                                          IMG[f] = p

def dedup(items):
    seen, out = set(), []
    for p, y in items:
        k = (y, os.path.basename(p))
        if k not in seen: seen.add(k); out.append((p, y))
    return out
FER_TR, FER_TE, CK = dedup(FER_TR), dedup(FER_TE), dedup(CK)
assert train_csv and test_csv and FER_TR and FER_TE, 'need RAF-DB and FER2013 as inputs'
assert ARCH != 'ddamfn' or ckpt_path, 'the DDAMFN member needs the RAF-DB checkpoint as an input'

raf_full = pd.read_csv(train_csv); raf_test_df = pd.read_csv(test_csv)
raf_tr_df, raf_va_df = train_test_split(raf_full, test_size=0.10, stratify=raf_full['label'], random_state=SPLIT_SEED)
to_items = lambda df: [(IMG[n], int(l) - 1) for n, l in zip(df['image'], df['label'])]
RAF_TR, RAF_VA, RAF_TE = to_items(raf_tr_df), to_items(raf_va_df), to_items(raf_test_df)
FER_TR, FER_VA = train_test_split(FER_TR, test_size=0.10, stratify=[y for _, y in FER_TR], random_state=SPLIT_SEED)
if os.environ.get('FAST'):
    RAF_TR, FER_TR = RAF_TR[:256], FER_TR[:256]
EVALS = {'raf_val': RAF_VA, 'raf_test': RAF_TE, 'fer_val': FER_VA, 'fer_test': FER_TE, 'ck': CK}
print(f'{ARCH} seed {SEED} | RAF train {len(RAF_TR)} | FER train {len(FER_TR)} | ' +
      ' | '.join(f'{k} {len(v)}' for k, v in EVALS.items()) + f' | {DEVICE} | {CFG}')

## 2 · Augmentation, sampler, model

In [ ]:
class LowRes:
    def __init__(self, p, lo=28, hi=64): self.p, self.lo, self.hi = p, lo, hi
    def __call__(self, img):
        if random.random() < self.p:
            s = random.randint(self.lo, self.hi)
            img = img.resize((s, s), Image.BILINEAR).resize((S, S), Image.BILINEAR)
        return img

NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((S, S)), transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.RandomRotation(12), transforms.RandomCrop(S, padding=S // 14)], p=0.5),
    transforms.RandomGrayscale(p=0.3), LowRes(p=0.3),
    transforms.ColorJitter(brightness=0.35, contrast=0.35),
    transforms.ToTensor(), NORM, transforms.RandomErasing(scale=(0.02, 0.25)),
])
eval_tf = transforms.Compose([transforms.Resize((S, S)), transforms.ToTensor(), NORM])

class Faces(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]; return self.tf(Image.open(p).convert('RGB')), y

w = [0.5 / len(RAF_TR)] * len(RAF_TR) + [0.5 / len(FER_TR)] * len(FER_TR)
g = torch.Generator().manual_seed(SEED)
sampler = WeightedRandomSampler(w, num_samples=2 * len(RAF_TR), replacement=True, generator=g)
train_loader = DataLoader(Faces(RAF_TR + FER_TR, train_tf), batch_size=CFG['batch'], sampler=sampler,
                          num_workers=CFG['workers'], pin_memory=True, drop_last=True)
def eval_loader(items): return DataLoader(Faces(items, eval_tf), batch_size=256, num_workers=CFG['workers'])

if ARCH == 'ddamfn':
    from DDAM import DDAMNet
    model = DDAMNet(num_class=7, num_head=2, pretrained=False).to(DEVICE)
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck.get('model_state_dict', ck))
    logits_of = lambda out: out[0]
else:
    import timm
    model = timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=not os.environ.get('FAST'), num_classes=7,
                              drop_path_rate=0.1).to(DEVICE)
    logits_of = lambda out: out

@torch.no_grad()
def predict(items):
    model.eval(); out = []
    for x, _ in eval_loader(items):
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            out.append(logits_of(model(x.to(DEVICE))).float().cpu())
    return torch.cat(out).numpy()

def acc(logits, items):
    y = np.array([l for _, l in items]); p = logits.argmax(1)
    return 100 * (p == y).mean(), 100 * balanced_accuracy_score(y, p)

## 3 · Train

In [ ]:
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
if ARCH == 'ddamfn':
    class AttentionLoss(nn.Module):
        def forward(self, heads):
            n = len(heads); loss = 0.0; cnt = 0
            if n > 1:
                for i in range(n - 1):
                    for j in range(i + 1, n):
                        loss = loss + F.mse_loss(heads[i], heads[j]); cnt += 1
                loss = cnt / (loss + 1e-7)
            return loss
    att = AttentionLoss()
    class SAM(torch.optim.Optimizer):
        def __init__(self, params, base_optimizer, rho=0.05, **kw):
            super().__init__(params, dict(rho=rho, **kw))
            self.base_optimizer = base_optimizer(self.param_groups, **kw); self.param_groups = self.base_optimizer.param_groups
        @torch.no_grad()
        def first_step(self):
            norm = torch.norm(torch.stack([p.grad.norm(2) for g in self.param_groups for p in g['params'] if p.grad is not None]), 2)
            for g in self.param_groups:
                for p in g['params']:
                    if p.grad is None: continue
                    self.state[p]['old_p'] = p.data.clone(); p.add_(p.grad * g['rho'] / (norm + 1e-12))
            self.zero_grad()
        @torch.no_grad()
        def second_step(self):
            for g in self.param_groups:
                for p in g['params']:
                    if p.grad is not None: p.data = self.state[p]['old_p']
            self.base_optimizer.step(); self.zero_grad()
    opt = SAM(model.parameters(), torch.optim.Adam, rho=0.05, lr=CFG['lr']); base_opt = opt.base_optimizer
    def train_step(x, y):
        def L():
            out, _, heads = model(x); return crit(out, y) + 0.1 * att(heads)
        loss = L(); loss.backward(); opt.first_step(); L().backward(); opt.second_step(); return loss.item()
else:
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=0.05); base_opt = opt
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    def train_step(x, y):
        opt.zero_grad()
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); return loss.item()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(base_opt, T_max=CFG['epochs'])

best, best_state = -1, None
for ep in range(1, CFG['epochs'] + 1):
    model.train(); t0 = time.time(); run = 0.0
    for x, y in train_loader:
        run += train_step(x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True))
    sched.step()
    rv, _ = acc(predict(RAF_VA), RAF_VA); fv, _ = acc(predict(FER_VA), FER_VA); score = (rv + fv) / 2
    flag = ''
    if score > best: best, best_state, flag = score, copy.deepcopy(model.state_dict()), '  * best'
    print(f'epoch {ep:2d}/{CFG["epochs"]} | loss {run / len(train_loader):.3f} | val RAF {rv:.2f}% | val FER {fv:.2f}% | {time.time() - t0:.0f}s{flag}')
model.load_state_dict(best_state)
torch.save({'model_state_dict': best_state, 'arch': ARCH, 'seed': SEED, 'class_names': CLASS_NAMES},
           os.path.join(OUT, f'member_{ARCH}_s{SEED}.pth'))

## 4 · Save logits on every evaluation set (keyed by image) and report

In [ ]:
key = lambda p, y: f'{y}/{os.path.basename(p)}'
save = {}
for name, items in EVALS.items():
    if not items: continue
    lg = predict(items); save[f'{name}_logits'] = lg
    save[f'{name}_keys'] = np.array([key(p, y) for p, y in items]); save[f'{name}_y'] = np.array([y for _, y in items])
    a, b = acc(lg, items); print(f'{name:9s} overall {a:6.2f}% | mean-class {b:6.2f}%')
np.savez_compressed(os.path.join(OUT, f'logits_{ARCH}_s{SEED}.npz'), **save)
print('saved', sorted(os.listdir(OUT)))